In [2]:
import sys
sys.path.append("../..")

import pandas as pd
from agronomic_functions import build_flat
from agronomic_functions import quality_flags
from db_config import run_query
from db_config import get_engine
engine = get_engine()

flat = build_flat("orgc", ["ph", "clay", "sand"])
flat.to_sql("flat_orgc_ghana", engine, if_exists="replace", index=False)
check = run_query("SELECT * FROM flat_orgc_ghana;")


# STEP B: The Test (Proving the alarm works)
# ==========================================
def test_quality_flags():
    # Build the 6-row fake table
    test_df = pd.DataFrame({
        "orgc": [50, 50, 50, -1, 300, 50],        
        "ph":   [7, 14.9, 7, 7, 7, float("nan")], 
        "clay": [30, 30, 60, 30, 30, 30],         
        "sand": [40, 40, 50, 40, 40, 40]          
    })
    
    result = quality_flags(test_df)
    
    # The Asserts (The Bouncer)
    assert result.loc[0].filter(like="flag").sum() == 0, "Row 0 should be clean"
    assert result.loc[1, "flag_ph"] == True, "Row 1 should flag pH > 14"
    assert result.loc[2, "flag_clay_sand"] == True, "Row 2 should flag clay+sand > 100"
    assert result.loc[3, "flag_oc_neg"] == True, "Row 3 should flag negative OC"
    assert result.loc[4, "flag_oc_high"] == True, "Row 4 should flag OC > 200"
    assert result.loc[5, "flag_ph"] == False, "Row 5 (missing pH) must NOT be flagged"
    
    print("all tests passed")

# Run the test
test_quality_flags()

# ==========================================
# STEP C: Predict and Run on the real 'flat' table
# ==========================================
# (Assuming 'flat' is loaded in your notebook already)
flat_checked = quality_flags(flat)

flag_counts = flat_checked[[
    "flag_ph", "flag_clay_sand", "flag_clay", 
    "flag_sand", "flag_oc_neg", "flag_oc_high"
]].sum()

print("Flag counts on real data:")
print(flag_counts)

meta = run_query("SELECT layer_id, dataset_id FROM wosis_orgc WHERE country_name = 'Ghana'")

# 2. Merge it in
traced = pd.merge(flat_checked, meta, on="layer_id", how="left")

# 3. Prove the row count is safe
print("Original rows:", len(flat_checked))
print("Merged rows:", len(traced))  # This MUST be 1756

# 4. Count high-carbon rows per dataset
high_carbon = traced[traced["flag_oc_high"] == True]
print("High carbon rows per dataset:")
print(high_carbon["dataset_id"].value_counts())


all tests passed
Flag counts on real data:
flag_ph             0
flag_clay_sand      0
flag_clay           0
flag_sand           0
flag_oc_neg         0
flag_oc_high      144
dtype: int64
Original rows: 1756
Merged rows: 1756
High carbon rows per dataset:
dataset_id
GH-GhaSP    144
Name: count, dtype: int64


In [ ]:
flat_checked

,profile_id,layer_id,orgc,ph,clay,sand,flag_ph,flag_clay_sand,flag_clay,flag_sand,flag_oc_neg,flag_oc_high
0,1144539,189,14.9,4.3,33.3,54.5,False,False,False,False,False,False
1,1144539,190,10.0,4.2,40.8,46.3,False,False,False,False,False,False
2,1144539,191,4.0,4.5,39.6,50.3,False,False,False,False,False,False
3,1144539,192,4.9,4.6,45.0,46.0,False,False,False,False,False,False
4,1144539,193,3.0,4.6,49.4,42.9,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1751,1516873,3091764,309.0,6.1,NaN,NaN,False,False,False,False,False,True
1752,1582495,107082,15.6,4.3,NaN,NaN,False,False,False,False,False,False
1753,1238868,3091564,98.9,6.9,NaN,NaN,False,False,False,False,False,False
1754,1308497,3091470,20.0,4.9,NaN,NaN,False,False,False,False,False,False


: 